In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.models.input_layer import (
    load_modeling_splits,
    InputConfig,
    TabularInputLayer,
    resolve_feature_columns,
)
from src.models.feature_sets import BASE_TABULAR_FEATURES
from src.evaluation.metrics import evaluate_binary_probabilities

In [2]:
splits = load_modeling_splits(dataset_name="earthquake_aftershock_v2_gcmt")

train_df = splits["train"]
val_df = splits["val"]
test_df = splits["test"]

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (14968, 92)
Val shape: (1461, 92)
Test shape: (2052, 92)


In [3]:
feature_cols = resolve_feature_columns(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    requested_cols=BASE_TABULAR_FEATURES,
    allow_missing_optional=True,
)

print("Using features:")
print(feature_cols)

Using features:
['trigger_latitude', 'trigger_longitude', 'trigger_depth_km', 'trigger_magnitude', 'trigger_month', 'trigger_dayofyear', 'trigger_hour', 'prior_global_event_count_24h', 'prior_global_event_count_7d']


In [8]:
def train_logreg_for_target(target_col, class_weight=None, C=1.0):
    config = InputConfig(
        feature_cols=feature_cols,
        target_col=target_col,
        missing_strategy="median",
        scale=True,
        drop_rows_with_missing_target=True,
        allow_missing_optional=True,
    )

    input_layer = TabularInputLayer()
    input_layer.fit(train_df, config)

    train_transformed = input_layer.transform(train_df)
    val_transformed = input_layer.transform(val_df)
    test_transformed = input_layer.transform(test_df)

    train_mask = train_df[target_col].notna()
    val_mask = val_df[target_col].notna()
    test_mask = test_df[target_col].notna()

    X_train = train_transformed.loc[train_mask, feature_cols].copy()
    y_train = train_df.loc[train_mask, target_col].astype(int).copy()

    X_val = val_transformed.loc[val_mask, feature_cols].copy()
    y_val = val_df.loc[val_mask, target_col].astype(int).copy()

    X_test = test_transformed.loc[test_mask, feature_cols].copy()
    y_test = test_df.loc[test_mask, target_col].astype(int).copy()

    model = LogisticRegression(
        max_iter=2000,
        class_weight=class_weight,
        C=C,
        solver="lbfgs",
        random_state=42,
    )

    model.fit(X_train, y_train)

    train_prob = model.predict_proba(X_train)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    train_metrics = evaluate_binary_probabilities(y_train, train_prob)
    val_metrics = evaluate_binary_probabilities(y_val, val_prob)
    test_metrics = evaluate_binary_probabilities(y_test, test_prob)

    metrics_df = pd.DataFrame([
        {"split": "train", "target": target_col, **train_metrics},
        {"split": "val", "target": target_col, **val_metrics},
        {"split": "test", "target": target_col, **test_metrics},
    ])

    pred_tables = {
        "train": pd.DataFrame({
            "trigger_event_id": train_df.loc[train_mask, "trigger_event_id"].values,
            "y_true": y_train.values,
            "y_prob": train_prob,
            "split": "train",
            "target": target_col,
            "model_name": "logreg_base",
        }),
        "val": pd.DataFrame({
            "trigger_event_id": val_df.loc[val_mask, "trigger_event_id"].values,
            "y_true": y_val.values,
            "y_prob": val_prob,
            "split": "val",
            "target": target_col,
            "model_name": "logreg_base",
        }),
        "test": pd.DataFrame({
            "trigger_event_id": test_df.loc[test_mask, "trigger_event_id"].values,
            "y_true": y_test.values,
            "y_prob": test_prob,
            "split": "test",
            "target": target_col,
            "model_name": "logreg_base",
        }),
    }

    return model, metrics_df, pred_tables

In [9]:
model_24h, metrics_24h, preds_24h = train_logreg_for_target("y_24h")
metrics_24h

,split,target,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,train,y_24h,0.218343,0.632975,0.701915,14968,0.438268
1,val,y_24h,0.211554,0.611540,0.697125,1461,0.390144
2,test,y_24h,0.203154,0.594477,0.772720,2052,0.524366


In [10]:
model_72h, metrics_72h, preds_72h = train_logreg_for_target("y_72h")
metrics_72h

,split,target,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,train,y_72h,0.228040,0.650656,0.676093,14968,0.501203
1,val,y_72h,0.227992,0.647669,0.661882,1461,0.450376
2,test,y_72h,0.208006,0.602938,0.746592,2052,0.587232


In [11]:
all_metrics = pd.concat([metrics_24h, metrics_72h], ignore_index=True)
all_metrics

,split,target,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,train,y_24h,0.218343,0.632975,0.701915,14968,0.438268
1,val,y_24h,0.211554,0.611540,0.697125,1461,0.390144
2,test,y_24h,0.203154,0.594477,0.772720,2052,0.524366
3,train,y_72h,0.228040,0.650656,0.676093,14968,0.501203
4,val,y_72h,0.227992,0.647669,0.661882,1461,0.450376
5,test,y_72h,0.208006,0.602938,0.746592,2052,0.587232


In [12]:
output_dir = REPO_ROOT / "reports" / "metrics"
output_dir.mkdir(parents=True, exist_ok=True)

all_predictions = pd.concat([
    preds_24h["train"], preds_24h["val"], preds_24h["test"],
    preds_72h["train"], preds_72h["val"], preds_72h["test"],
], ignore_index=True)

all_metrics.to_csv(output_dir / "logreg_baseline_metrics.csv", index=False)
all_predictions.to_csv(output_dir / "logreg_baseline_predictions.csv", index=False)

print("Saved:")
print(output_dir / "logreg_baseline_metrics.csv")
print(output_dir / "logreg_baseline_predictions.csv")

Saved:
/Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/logreg_baseline_metrics.csv
/Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/logreg_baseline_predictions.csv
